# 15. 자료와 표본 독립감사

**감사 질문:** 전처리 계산이 맞더라도 점포의 정의, 관측 가능한 상권의 구성, 분기 비교가 연구 질문을 지지하는가?

종로구·마포구의 공개 외식업 10종, 2023~2025년 분기를 대상으로 한다. 기존 산출물을 읽기만 하고 이 노트북과 `reports/20261005/audit_20261005/sample/`만 쓴다. 원자료 미제공 매출은 0으로 바꾸지 않는다. 모든 실행은 오프라인이며 새 커널에서 위→아래 순서로 실행한다. API 수집 및 기존 분석 수정은 하지 않는다.

**판정 구분:** (1) 파일·계산 무결성, (2) 측정 정의 및 표본 대표성, (3) 아직 확인되지 않은 원천 제공 방식. 실제 관측된 차이와 그 원인에 대한 추정을 구분한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import json, hashlib, re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = PROJECT_ROOT
OUT = project_path('reports/20261005/audit_20261005/sample')
OUT.mkdir(parents=True, exist_ok=True)
KEY = ['quarter','area_id','industry_id']
DTYPES = {k:'string' for k in KEY}
panel = pd.read_csv(project_path('dataset/analysis/food_industry_panel.csv'), dtype=DTYPES)
areas = pd.read_csv(project_path('dataset/analysis/study_areas.csv'), dtype={'area_id':'string','district_code':'string'})
manifest = project_json_loads((project_path('metadata/seoul_analysis_sources.json')).read_text(encoding='utf-8'))
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 24)
pd.set_option('display.float_format', lambda x:f'{x:,.3f}')
def sha(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def save(name, frame):
    frame.to_csv(OUT/(name+'.csv'), index=False, encoding='utf-8-sig')
    return frame
tracked = [project_path('dataset/analysis/food_industry_panel.csv'), project_path('dataset/analysis/study_areas.csv'), project_path('metadata/seoul_analysis_sources.json')]
initial_hash = {p.relative_to(ROOT).as_posix():sha(p) for p in tracked}
assert manifest['validation_passed'] and manifest['status']=='validated'
assert not panel.duplicated(KEY).any()
print('관측 기간:', panel.quarter.min(), '부터', panel.quarter.max(), '/ 격자:',len(panel))

## 1. 관측 기간과 수집 시점을 따로 확인

2026년 재수집 날짜는 2026년 매출을 관측했다는 뜻이 아니다. 등록된 6개 원파일의 해시·키·실제 분기 집합을 확인하고, 현재 상권 목록과 매년 관측된 상권 ID가 어떻게 겹치는지 기록한다. 원자료 전체 행을 읽되 조사 값은 필요한 열만 유지한다.

In [ ]:
rename = {
 '기준_년분기_코드':'quarter','상권_코드':'area_id','서비스_업종_코드':'industry_id',
 '상권_코드_명':'raw_area_name','상권_구분_코드':'area_type','서비스_업종_코드_명':'industry',
 '당월_매출_금액':'total_sales','당월_매출_건수':'sales_transactions','시간대_21~24_매출_금액':'night_sales',
 '점포_수':'store_count','유사_업종_점포_수':'similar_industry_store_count','개업_점포_수':'opened_stores',
 '폐업_점포_수':'closed_stores','프랜차이즈_점포_수':'franchise_stores',
 'stdr_yyqu_cd':'quarter','trdar_cd':'area_id','svc_induty_cd':'industry_id','trdar_cd_nm':'raw_area_name',
 'trdar_se_cd':'area_type','svc_induty_cd_nm':'industry','stor_co':'store_count',
 'similr_induty_stor_co':'similar_industry_store_count','opbiz_stor_co':'opened_stores',
 'clsbiz_stor_co':'closed_stores','frc_stor_co':'franchise_stores'}
raw_frames, provenance = {'sales':[],'stores':[]}, []
for item in manifest['items']:
    kind = item['dataset']
    if kind not in raw_frames: continue
    entry = next(e for e in item['extracted'] if e['path'].lower().endswith('.csv'))
    path = project_path(entry['path'])
    assert sha(path)==entry['sha256']
    initial_hash[entry['path']]=entry['sha256']
    f = pd.read_csv(path, encoding='cp949', usecols=lambda x:x in rename).rename(columns=rename)
    for col in KEY: f[col]=f[col].astype('string')
    provenance.append(dict(dataset=kind, file=entry['path'], file_year=item['year'],
        rows=len(f), duplicate_keys=int(f.duplicated(KEY).sum()), missing_keys=int(f[KEY].isna().any(axis=1).sum()),
        actual_quarters=','.join(sorted(f.quarter.unique())), observed_area_ids=f.area_id.nunique(),
        current_study_ids=f.loc[f.area_id.isin(areas.area_id),'area_id'].nunique(), sha256=entry['sha256']))
    raw_frames[kind].append(f)
raw_sales=pd.concat(raw_frames['sales'],ignore_index=True)
raw_stores=pd.concat(raw_frames['stores'],ignore_index=True)
source_profile=save('01_source_profile',pd.DataFrame(provenance))
display(source_profile.drop(columns=['file','sha256']))

**해석:** 중복 0과 4개 분기 존재는 파일 내부 일관성이다. 비공개·미제공 상권의 누락, 표준단위 변경 이후 과거 값의 소급 재집계 여부, 당시 위치까지 증명하지 않는다. 01 셀 5·7은 초기 원파일·현재 경계, 08 셀 3·6·10은 검증 매니페스트와 기존 상권 목록을 사용한다.

## 2. 점포 수 분모를 독립적으로 재해석

서울시 점포 공식 페이지는 **전체 점포 수 = 일반 점포 수 + 프랜차이즈 점포 수**라고 설명한다. 기존 01 셀 5·11, 08 셀 5·12는 `점포_수/stor_co`를 `store_count`로 쓰고 `유사_업종_점포_수`를 제외한다. 먼저 원파일에서 세 열의 항등식을 전수 확인한다.

공식 근거: [서울시 점포 자료](https://data.seoul.go.kr/dataList/OA-15577/S/1/datasetView.do), 2026-10-05 확인. 보관 HTML: `metadata/seoul_stores_page.html` (설명 39행/1345행). 매출 공식 설명만으로 매출 추정 모집단이 일반·프랜차이즈 중 어느 범위인지 확정할 수 없으므로 그 부분은 미확인으로 남긴다.

In [ ]:
store_identity = raw_stores.assign(identity_ok=lambda f:f.similar_industry_store_count.eq(f.store_count+f.franchise_stores))
identity_table=save('02_store_identity',store_identity.groupby('quarter',observed=True).agg(
    source_rows=('area_id','size'), equal_regular_plus_franchise=('identity_ok','sum'),
    regular_zero=('store_count',lambda s:s.eq(0).sum()), franchise_positive=('franchise_stores',lambda s:s.gt(0).sum())).reset_index())
zero_sales=panel.loc[panel.total_sales.gt(0)&panel.store_count.eq(0)].copy()
zero_summary=save('03_zero_regular_store_sales',zero_sales.groupby(['industry_id','industry'],observed=True).agg(
    rows=('area_id','size'), areas=('area_id','nunique'), franchise_positive=('franchise_stores',lambda s:s.gt(0).sum()),
    all_store_positive=('similar_industry_store_count',lambda s:s.gt(0).sum()), sales_sum=('total_sales','sum')).reset_index())
save('03_zero_regular_store_rows',zero_sales)
display(identity_table)
display(zero_summary)
print('매출 양수·일반점포 0:',len(zero_sales),'/ 전체점포도 0:',int(zero_sales.similar_industry_store_count.eq(0).sum()))

**확정 결과:** 전체 919,405 원행에서 `유사_업종_점포_수 = 점포_수 + 프랜차이즈_점포_수`가 성립한다. 양수 매출·일반점포 0인 95행은 **모두 프랜차이즈가 양수**이고 전체점포 0인 행은 없다. 제과 17행·패스트푸드 24행·치킨 47행·커피 7행이다. 따라서 이를 ‘점포 자체가 없는데 매출이 존재하는 95개 자료 불일치’라고 설명하는 것은 잘못이다. `store_count`를 일반점포 수로 명시하고, 전체점포 수와 분리해야 한다.

매출 모집단에 프랜차이즈가 어떻게 포함되는지는 현재 공식 설명만으로 확정하지 않았다. 분모 교체에 앞서 공급기관의 매출 추정 정의를 확인해야 한다. 총매출과 야간비중만 쓰는 **09 기본모형의 결과가 이 변수명 오류만으로 틀렸다는 뜻은 아니다.** 직접 영향을 받는 것은 점포당 매출, 점포 규모 설명, 09/10의 점포 통제 및 95행 경고이다.

점포당 매출 및 점포 수 통제의 영향 범위를 아래에서 정량화한다. 기존 값을 덮어쓰거나 총수로 자동 교정하지 않고 두 정의의 차이만 계산한다.

In [ ]:
denom=panel.loc[panel.total_sales.notna()&panel.store_count.gt(0)&panel.similar_industry_store_count.gt(0)].copy()
denom['ratio_regular_to_all'] = denom.similar_industry_store_count/denom.store_count
denom['franchise_share'] = denom.franchise_stores/denom.similar_industry_store_count
denominator_impact=save('04_denominator_impact',denom.groupby(['industry_id','industry'],observed=True).agg(
    rows=('area_id','size'), franchise_present_rows=('franchise_stores',lambda s:s.gt(0).sum()),
    median_ratio=('ratio_regular_to_all','median'), p90_ratio=('ratio_regular_to_all',lambda s:s.quantile(.9)),
    max_ratio=('ratio_regular_to_all','max'), median_franchise_share=('franchise_share','median')).reset_index())
display(denominator_impact)
display(denom.nlargest(8,'ratio_regular_to_all')[KEY+['area_name','industry','store_count','franchise_stores','similar_industry_store_count','ratio_regular_to_all']])

**확정 결과:** 동일 매출의 ‘일반점포 분모 비율/전체점포 분모 비율’ 중앙값은 치킨 2.143배, 패스트푸드 1.646배, 제과 1.250배이며 최대 9배다. 이는 프랜차이즈 비중이 업종별로 달라 점포당 매출의 업종 비교가 특히 달라질 수 있음을 뜻한다. 어느 비율이 실제 점포 평균 매출인지는 매출 모집단 확인이 필요하다. 일반점포 0인 행을 일괄 제거하는 민감도도 프랜차이즈 중심 상권을 선택적으로 뺀다.

## 3. ‘매출 미제공’을 업종 부재로 볼 수 있는가?

전체 격자의 매출 제공 여부를 점포 자료 유무·일반/전체 점포 규모별로 나눈다. 0 일반점포와 0 전체점포를 구분하고, 점포 자료가 없는 행은 별도 범주로 남긴다.

In [ ]:
panel['all_store_band']=pd.cut(panel.similar_industry_store_count,[-.1,0,1,2,5,10,20,50,np.inf],
    labels=['0','1','2','3–5','6–10','11–20','21–50','51+']).astype('string').fillna('자료없음')
missing_by_size=save('05_sales_availability_by_store_size',panel.groupby('all_store_band',observed=True).agg(
    expected_rows=('area_id','size'), sales_rows=('sales_record_present','sum'),
    median_activity=('known_performance_count_500m','median')).reset_index())
missing_by_size['availability_pct']=100*missing_by_size.sales_rows/missing_by_size.expected_rows
save('05_sales_availability_by_store_size',missing_by_size)
missing_positive=panel.loc[~panel.sales_record_present&panel.similar_industry_store_count.gt(0)].copy()
missing_industry=save('06_missing_with_positive_stores',missing_positive.groupby(['industry_id','industry'],observed=True).agg(
    rows=('area_id','size'), areas=('area_id','nunique'), median_all_stores=('similar_industry_store_count','median'),
    max_all_stores=('similar_industry_store_count','max')).reset_index())
display(missing_by_size)
display(missing_industry)
print('미제공',int((~panel.sales_record_present).sum()),'중 전체점포 양수',len(missing_positive))

**확정 결과:** 매출 미제공 10,459행 중 5,564행은 전체점포 수가 양수다. 전체점포 1~2개인 **4,430행의 제공률은 0%**이고, 3~5개 66.6%, 6~10개 92.5%, 11~20개 96.3%, 21~50개 99.8%, 51개 이상 100%다. 이는 제공되는 표본이 점포 규모에 강하게 선택된다는 직접 증거다. 소수점포 비공개 규칙이 원인이라는 해석은 추정이며 별도 공식 근거가 필요하다. 매출이 결측인지 여부가 잠재적 매출 자체에 의존하는 MNAR인지는 확정하지 않는다.

이 문제는 결측을 0으로 채우지 않고 누락률을 보고하는 것만으로 해소되지 않는다. 관측 표본의 결과를 작은 상권·업종, 종로·마포 전체 또는 업종의 평균 관계로 일반화할 수 없으며 분석 대상 정의를 ‘매출이 공개되는 상권×업종’으로 제한해야 한다.

## 4. 업종별 표본과 공통 지지집합

09 기본모형의 실제 관측 키를 읽는다. 최소 2분기 조건 적용 후 업종별 상권·공연 분포를 비교하고, 모든 업종이 같은 상권·분기에 관측되는 교집합을 별도로 계산한다. 공통 표본으로 제한하더라도 인과 식별이 생기지는 않는다.

In [ ]:
membership=pd.read_csv(project_path('reports/20261005/09_model_sample_membership.csv'),dtype=DTYPES)
main_keys=membership.loc[membership.scenario.eq('main_500')&membership.outcome.eq('log_sales'),KEY]
main=panel.merge(main_keys,on=KEY,how='inner',validate='one_to_one')
industry_support=save('07_industry_support',main.groupby(['industry_id','industry'],observed=True).agg(
    rows=('area_id','size'), areas=('area_id','nunique'),
    zero_activity_pct=('known_performance_count_500m',lambda s:100*s.eq(0).mean()),
    median_activity=('known_performance_count_500m','median'), mean_activity=('known_performance_count_500m','mean'),
    activity_p90=('known_performance_count_500m',lambda s:s.quantile(.9)),
    median_all_stores=('similar_industry_store_count','median')).reset_index())
wide=main.assign(observed=1).pivot(index=['area_id','quarter'],columns='industry_id',values='observed')
common_keys=wide.loc[wide.notna().all(axis=1)].reset_index()[['area_id','quarter']]
common=main.merge(common_keys,on=['area_id','quarter'],how='inner',validate='many_to_one')
common_complete=common.groupby('area_id').quarter.nunique()
common_summary=pd.DataFrame([dict(main_rows=len(main), main_areas=main.area_id.nunique(),
    main_unique_area_quarters=len(wide), all10_area_quarters=len(common_keys),
    all10_areas=common.area_id.nunique(), all10_all12_areas=int(common_complete.eq(12).sum()),
    all10_rows=len(common), main_rows_retained_pct=100*len(common)/len(main))])
save('08_common_support_summary',common_summary)
save('08_common_support_keys',common_keys)
pair=[]
for a in wide.columns:
    for b in wide.columns:
        both=wide[[a,b]].notna().all(axis=1) if a!=b else wide[a].notna()
        pair.append(dict(industry_a=a,industry_b=b,common_area_quarters=int(both.sum()),
            common_areas=wide.index.get_level_values('area_id')[both].nunique(),
            overlap_fraction_a=float(both.sum()/wide[a].notna().sum())))
save('09_pairwise_support',pd.DataFrame(pair))
display(industry_support)
display(common_summary)

**확정 결과:** 기본모형 7,646행·134상권 중 10종 공통표본은 **16상권·170상권분기·1,700행(22.23%)**, 12분기 모두 10종이 관측되는 공통표본은 **12상권**이다. 업종별 공연 수 중앙값도 한식 20개, 일식 59개처럼 다르다.

업종별 기울기는 서로 다른 상권·노출범위를 가중한 조건부 관계다. 상권 군집 공분산은 같은 상권의 업종 간 오차 관련성을 처리하지만 표본 자체를 일치시키지는 않는다. 따라서 업종 간 동일기울기 검정은 ‘같은 장소에서 업종만 달랐을 때 차이’와 일치하지 않을 수 있다. 공통표본은 작은 번화 상권으로 또 선택될 수 있으므로 전체모형을 대체하기보다 별도 민감도로 보고해야 한다.

## 5. 균형표본은 고정 사업체 표본인가?

12분기 모두 매출이 제공되는 **상권×업종**과 일부만 제공되는 조합의 규모를 비교한다. 매출 제공의 진입·이탈을 실제 사업체 개폐업과 구분하고, 균형표본에도 개폐업이 있는지 확인한다.

In [ ]:
entities=panel.groupby(['area_id','industry_id'],observed=True).agg(sales_quarters=('sales_record_present','sum')).reset_index()
panel=panel.merge(entities,on=['area_id','industry_id'],validate='many_to_one')
panel['coverage_group']=np.select([panel.sales_quarters.eq(12),panel.sales_quarters.eq(0)],['12분기 제공','항상 미제공'],default='일부 분기 제공')
cohort_summary=save('10_cohort_summary',panel.groupby('coverage_group',observed=True).agg(
    entity_rows=('area_id','size'), sales_rows=('sales_record_present','sum'),
    median_all_stores=('similar_industry_store_count','median'), median_activity=('known_performance_count_500m','median'),
    median_observed_sales=('total_sales','median'),
    rows_with_openings=('opened_stores',lambda s:s.gt(0).sum()), rows_with_closures=('closed_stores',lambda s:s.gt(0).sum())).reset_index())
cohort_summary['area_industry_entities']=cohort_summary.entity_rows/12
save('10_cohort_summary',cohort_summary)
ordered=panel.sort_values(['area_id','industry_id','quarter']).copy()
g=ordered.groupby(['area_id','industry_id'],observed=True)
ordered['previous_sales_present']=g.sales_record_present.shift()
ordered['previous_all_stores']=g.similar_industry_store_count.shift()
ordered['availability_transition']=np.select([
 ordered.previous_sales_present.eq(True)&~ordered.sales_record_present,
 ordered.previous_sales_present.eq(False)&ordered.sales_record_present],['제공→미제공','미제공→제공'],default='유지/첫분기')
transitions=ordered.loc[ordered.availability_transition.ne('유지/첫분기')].copy()
transition_summary=save('11_availability_transitions',transitions.groupby('availability_transition',observed=True).agg(
    rows=('area_id','size'), current_positive_stores=('similar_industry_store_count',lambda s:s.gt(0).sum()),
    previous_positive_stores=('previous_all_stores',lambda s:s.gt(0).sum()),
    zero_reported_openings=('opened_stores',lambda s:s.eq(0).sum()), zero_reported_closures=('closed_stores',lambda s:s.eq(0).sum())).reset_index())
save('11_availability_transition_rows',transitions)
display(cohort_summary)
display(transition_summary)

**확정 결과:** 585개 균형 조합의 전체점포 중앙값은 12개·관측매출 중앙값 4.112억원이며 일부 분기 제공 108개 조합은 3개·0.416억원이다. 12분기 균형표본에도 개업 양수 2,922행·폐업 양수 3,009행이 있다. 매출 제공→미제공 87건 및 미제공→제공 65건 **모두 이전·현재 전체점포는 양수**다.

균형표본은 안정적으로 공개되는 상권×업종을 고른다. 내부 사업체가 3년간 같다는 뜻이 아니며, 사업체별 개폐업·생존 효과를 이 자료로 추적할 수 없다. 관측이 끊기는 것을 업종 소멸로, 관측이 시작되는 것을 실제 개업으로 해석하면 안 된다. 균형표본 민감도는 이러한 규모 선택을 강화할 수 있다.

## 6. 분기별 급변과 점포·거래량의 동행

동일 상권×업종의 연속한 분기에서 매출 배율을 계산한다. 10배 이상/0.1배 이하를 감사용 경보로 고정하며 참값 오류라고 자동 판정하지 않는다. 규모 불일치, 점포 변화, 제공 알고리즘 변화, 실제 수요 충격을 구별할 후속 근거가 필요하다.

In [ ]:
ordered['q_index']=ordered.quarter.str[:4].astype(int)*4+ordered.quarter.str[-1].astype(int)
for value in ['total_sales','sales_transactions','store_count','similar_industry_store_count','night_share','known_performance_count_500m','q_index']:
    ordered['previous_'+value]=ordered.groupby(['area_id','industry_id'],observed=True)[value].shift()
paired=ordered.loc[(ordered.q_index-ordered.previous_q_index).eq(1)&ordered.total_sales.gt(0)&ordered.previous_total_sales.gt(0)].copy()
paired['sales_ratio']=paired.total_sales/paired.previous_total_sales
paired['transaction_ratio']=paired.sales_transactions/paired.previous_sales_transactions
paired['all_store_change']=paired.similar_industry_store_count-paired.previous_similar_industry_store_count
paired['extreme_10x']=paired.sales_ratio.ge(10)|paired.sales_ratio.le(.1)
paired['abs_log_change']=np.abs(np.log(paired.sales_ratio))
paired['night_share_change_pp']=100*(paired.night_share-paired.previous_night_share)
quarter_changes=save('12_quarter_changes',paired.groupby('quarter',observed=True).agg(
    consecutive_pairs=('area_id','size'), extreme_pairs=('extreme_10x','sum'), median_ratio=('sales_ratio','median'),
    median_abs_log_change=('abs_log_change','median'), p90_abs_log_change=('abs_log_change',lambda s:s.quantile(.9))).reset_index())
extreme=paired.loc[paired.extreme_10x].sort_values('abs_log_change',ascending=False)
save('13_extreme_quarter_rows',extreme)
display(quarter_changes)
display(extreme[KEY+['area_name','industry','previous_total_sales','total_sales','sales_ratio','transaction_ratio','all_store_change']].head(16))
print('10배 급변:',len(extreme),'/',len(paired),'중 전체점포수 동일:',int(extreme.all_store_change.eq(0).sum()))

**확정 결과:** 연속 양수매출 6,954쌍 중 20쌍이 10배 이상/0.1배 이하 변화이며 16쌍은 전체점포 수가 그대로다. 예를 들어 마포역 4번 양식은 2023Q2 약 9.917억원에서 Q3 92.8만원, 사직공원 한식은 2023Q2 2,617.8만원에서 Q3 3.3만원으로 바뀐다. 2024Q1 전환에는 이 기준의 극단 변화가 0쌍이다.

극단 변화는 오류 판정용이 아닌 영향도 점검 대상으로 저장한다. 2024년 경계 변경 안내를 모든 급변의 원인으로 단정할 수 없고, 실제 수요 변화와 추정 과정의 변화를 현 자료로 구분하지 못한다. 원자료 그대로 사용했다는 검사는 이 숫자의 경제적 타당성을 보증하지 않는다.

## 7. 현재 상권과 과거 분기의 대응

현재 경계에 없는 원자료 ID와 해마다 연구 ID의 지원 여부를 확인한다. 현재 상권 ID가 과거에도 존재한다는 사실과 당시 폴리곤의 동일성은 별개다. 현재 경계만으로 탈락한 옛 상권의 자치구를 복원할 수 없으면 그 수를 종로·마포의 누락이라고 부르지 않는다.

In [ ]:
import shapefile
boundary_item=next(i for i in manifest['items'] if i['dataset']=='areas')
shp_entry=next(e for e in boundary_item['extracted'] if e['path'].endswith('.shp'))
shp=project_path(shp_entry['path'])
reader=shapefile.Reader(str(shp), encoding=shp.with_suffix('.cpg').read_text().strip())
current_attributes=pd.DataFrame([r.as_dict() for r in reader.iterRecords()])
current_ids=set(current_attributes.TRDAR_CD.astype(str))
raw_area_coverage=[]
for kind,f in [('sales',raw_sales),('stores',raw_stores)]:
    for quarter,q in f.groupby('quarter',observed=True):
        ids=set(q.area_id)
        raw_area_coverage.append(dict(dataset=kind,quarter=quarter,source_ids=len(ids),
            outside_current_boundary=len(ids-current_ids),current_study_ids_with_any_industry=len(ids&set(areas.area_id)),
            current_study_ids_with_food=q.loc[q.industry_id.str.startswith('CS100')&q.area_id.isin(areas.area_id),'area_id'].nunique()))
area_coverage=save('14_raw_area_coverage',pd.DataFrame(raw_area_coverage))
area_versions=pd.concat([raw_sales[['area_id','raw_area_name','area_type']],raw_stores[['area_id','raw_area_name','area_type']]]).drop_duplicates()
name_counts=area_versions.groupby('area_id').raw_area_name.nunique()
name_changes=area_versions.loc[area_versions.area_id.isin(name_counts[name_counts.gt(1)].index)]
save('14_area_label_variants',name_changes)
display(area_coverage)
print('연도간 이름이 여러 개:',len(name_counts[name_counts.gt(1)]),'상권')

**확정된 범위:** 12분기 점포 원자료에는 매 분기 같은 1,650개 상권 ID가 있고 모두 현재 경계 ID에 포함된다. 따라서 이 파일들에서 ‘2024년 ID가 대량 교체됐다’는 주장은 지지되지 않는다.

[서울시 매출 공식 페이지](https://data.seoul.go.kr/dataList/OA-15572/S/1/datasetView.do)는 2024년 표준단위구역 변경과 2026-07-03 이후 2021년 이상 제공 범위를 안내한다. 이는 2023/2024에 이번 패널이 실제 단절되었다는 증거는 아니다. 2023년 값이 새 구역으로 소급 재집계되었는지, 현재 상권 폴리곤과 분기별 집계구가 일치하는지 확인할 메타데이터/역사 경계가 없다. 따라서 **시간·공간 측정의 동등성 미확인**으로 판정한다.

## 8. 판정용 핵심 수치와 변경 방지

측정 정의와 선택적 제공은 숫자로 판정하고, 매출 추정 모집단 및 역사 경계는 미확인 항목으로 남긴다. 행 수/키 무결성 통과를 데이터 타당성 전체 통과로 확대하지 않는다.

In [ ]:
audit_summary={
 'scope':'종로구·마포구/외식업10종/2023~2025분기',
 'network_requests_in_notebook':0,'panel_rows':len(panel),'observed_sales_rows':int(panel.sales_record_present.sum()),
 'raw_store_rows':len(raw_stores),'store_identity_all_rows':bool(store_identity.identity_ok.all()),
 'positive_sales_zero_regular_store_rows':len(zero_sales),
 'positive_sales_zero_regular_store_but_positive_franchise_rows':int(zero_sales.franchise_stores.gt(0).sum()),
 'positive_sales_zero_all_store_rows':int(zero_sales.similar_industry_store_count.eq(0).sum()),
 'missing_sales_positive_all_store_rows':len(missing_positive),
 'common_support':common_summary.to_dict('records')[0],
 'extreme_consecutive_sales_pairs':len(extreme),'consecutive_sales_pairs':len(paired),
 'extreme_pairs_unchanged_all_store_count':int(extreme.all_store_change.eq(0).sum()),
 'historical_boundary_equivalence_verified':False,'sales_store_population_equivalence_verified':False,
 'source_sha256':initial_hash}
assert all(sha(project_path(name))==value for name,value in initial_hash.items())
(OUT/'summary.json').write_text(json.dumps(audit_summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([{k:v for k,v in audit_summary.items() if not isinstance(v,dict)}]))
print('입력 해시 불변 확인:',len(initial_hash),'파일')

## 9. 우선순위별 판정

기존 분석을 고치지 않은 독립 감사다. 즉시 고칠 수 있는 용어·정의 문제와 추가 자료가 필요한 타당성 문제를 구분한다. 표의 셀은 0부터 시작하는 노트북 내부 인덱스다.

In [ ]:
findings = [
 dict(id='S1',priority='P1',status='정의·표식 오류 확정 / 매출 모집단 미확인',
      issue='일반점포를 전체 점포처럼 설명하고 95행을 자료 불일치로 해석',
      evidence='919,405원행 항등식 성립; 95/95행 프랜차이즈 양수; 전체점포0·양매출 0행',
      impact='점포당 매출·업종 규모 설명·점포수 통제·95행 경고; 기본 비통제 총매출/야간비중 모형은 직접 영향 없음',
      action='일반·프랜차이즈·전체점포 명시; 매출 추정 모집단 확인 후 분모/통제 재정의',
      existing_cells='01:5,11;08:5,12;09:1,3,15',audit_cells='5,7',source='https://data.seoul.go.kr/dataList/OA-15577/S/1/datasetView.do'),
 dict(id='S2',priority='P1',status='선택적 관측과 지원 차이 확정 / 편향 크기 미확인',
      issue='소규모 조합 미제공과 업종별 다른 장소를 비교하는 추정대상',
      evidence='총점포1~2개4,430행 제공0%; 미제공 중 점포양수5,564행; 10종공통16상권·1,700행(22.23%)',
      impact='151상권 전체/업종 자체의 차이로 일반화할 수 없음; 균형표본도 규모선택',
      action='대상을 매출 공개 조합으로 제한하고 공통표본 결과/표본구성 함께 보고; 임의 결측0·기계적 가중 금지',
      existing_cells='08:16~21;09:3,15,17',audit_cells='9,11,13',source='고정 분석패널과09 모델표본'),
 dict(id='S3',priority='P2',status='비교동등성 미확인 / 2024 단절 확정 아님',
      issue='표준단위 변경 전후·현재 폴리곤과 과거 매출 집계구의 대응 미검증',
      evidence='공식2024공간단위변경 공지; 동일1650점포상권ID 유지; 역사폴리곤/소급집계 설명 없음',
      impact='시간변화가 경제현상과 동일 정의로 측정되는지 보증할 수 없음',
      action='분기별경계 및2023소급집계 여부 확인; 불가능하면 현재경계 적용의 한계 명시',
      existing_cells='01:7;08:3,24',audit_cells='17,18',source='https://data.seoul.go.kr/dataList/OA-15572/S/1/datasetView.do'),
 dict(id='S4',priority='P2',status='급변 확정 / 원천오류 미확정',
      issue='점포가 고정인데도 일부 매출이 수십~수백배 급변',
      evidence='6,954연속관측 중20쌍10배이상/0.1배이하;16쌍전체점포동일',
      impact='극단 변동이 기울기에 미치는 영향 및 추정매출의 실질 의미 점검 필요',
      action='원행 추적과 영향도 민감도; 오류 근거 없이 삭제·대체하지 않음',
      existing_cells='08:12;09:3',audit_cells='15',source='검증된2023~2025원자료')
]
(OUT/'findings.json').write_text(json.dumps(findings,ensure_ascii=False,indent=2),encoding='utf-8')
display(save('15_findings',pd.DataFrame(findings)).drop(columns=['source','existing_cells','audit_cells']))